# 红利ETF量化分析系统 - 交互式演示

本 Notebook 展示系统的 5 大核心模块用法。

In [ ]:
# 环境准备
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.data.fetcher import DataFetcher
from src.analysis.equity_bond_spread import EquityBondSpread
from src.analysis.macro_state import MacroStateModel
from src.analysis.volatility import VolatilityModel
from src.analysis.risk_management import RiskManager
from src.analysis.grid_optimizer import GridOptimizer
from src.strategies.timing_system import TimingSystem
from src.utils.helpers import (
    print_header, print_metrics, plot_spread_history,
    plot_backtest_result, plot_volatility_analysis, plot_risk_heatmap
)

fetcher = DataFetcher()
print('✅ 模块加载成功')

---
## 模块1: 股债利差择时系统
**核心逻辑**: 红利ETF股息率 - 十年期国债收益率 → 滚动百分位 → 仓位信号

In [ ]:
print_header('股债利差择时')

start_date = DataFetcher.get_start_date(10)

# 获取数据
div_df = fetcher.get_index_dividend_yield('000922', start_date)
bond_df = fetcher.get_bond_yield(start_date)

# 计算利差
spread = EquityBondSpread()
spread.compute_spread(div_df, bond_df)
spread.compute_percentile()
signal = spread.generate_signal()

# 当前信号
latest = signal.iloc[-1]
print(f'\n📅 {latest["date"].strftime("%Y-%m-%d")}')
print(f'  利差: {latest["spread"]:.2%}')
print(f'  历史分位: {latest["percentile"]:.1%}')
print(f'  建议仓位: {latest["position"]:.0%}')
print(f'  信号: {latest["signal"]}')

# 回测
index_df = fetcher.get_index_daily('000922', start_date)
bt = spread.backtest(index_df)
metrics = spread.get_metrics(bt)
print_metrics(metrics)

---
## 模块2: 宏观状态识别 (HMM)
**核心**: 使用隐含马尔可夫模型划分经济四象限

In [ ]:
print_header('宏观状态识别')

hmm_model = MacroStateModel()
start_year = pd.Timestamp.now().year - 10

try:
    m1m2 = fetcher.get_m1_m2_gap(start_year)
    pmi = fetcher.get_macro_data('PMI', start_year)
    cpi = fetcher.get_macro_data('CPI', start_year)
    bond = fetcher.get_bond_yield(f'{start_year}0101')
    
    features, features_norm = hmm_model.prepare_features(m1m2, pmi, cpi, bond)
    hmm_model.train(features_norm)
    labeled = hmm_model.label_states(features)
    
    state_id, state_label = hmm_model.get_current_state()
    suggestion = hmm_model.suggest_position(state_label)
    
    print(f'\n当前宏观状态: {state_label}')
    print(f'建议仓位: {suggestion["position"]:.0%} - {suggestion["comment"]}')
    
    print('\n状态转移概率:')
    print(hmm_model.state_transition_matrix())
    
    print('\n各状态频率:')
    print(labeled['state_label'].value_counts())
    
except Exception as e:
    print(f'❌ 数据获取失败: {e}')
    print('提示: 请检查网络连接或使用VPN')

---
## 模块3: 波动率建模 (GARCH)
**核心**: GARCH(1,1)预测波动率，识别极端加仓/减仓信号

In [ ]:
print_header('波动率建模')

start_date = DataFetcher.get_start_date(5)
etf_data = fetcher.get_etf_daily('510880', start_date)
returns = etf_data['close'].pct_change().dropna()

vol_model = VolatilityModel()
params = vol_model.fit_garch(returns * 100)

print('GARCH(1,1)参数:')
for k, v in params.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

# 极端事件检测
events = vol_model.detect_extreme_events()
extreme = events[events['signal'] != '正常']
if len(extreme) > 0:
    last = extreme.index[-1]
    print(f'\n⚠️ 最近极端事件: {last.strftime("%Y-%m-%d")}')
    print(f'  Z-Score: {extreme.loc[last, "vol_zscore"]:.2f}')
    print(f'  信号: {extreme.loc[last, "signal"]}')

---
## 模块4: 风险管理
**核心**: VaR(Cornish-Fisher修正)、ES、EVT极值理论

In [ ]:
print_header('风险管理')

risk = RiskManager()
report = risk.full_risk_report(returns, holding=1_000_000)
print_metrics(report)

print('\nEVT极值理论分析:')
evt = risk.evt_gev(returns)
for k, v in evt.items():
    print(f'  {k}: {v}')

---
## 模块5: 综合择时系统
**核心**: 融合利差信号 + 宏观状态 + 波动率信号 → 统一仓位建议

In [ ]:
print_header('综合择时系统')

timing = TimingSystem()

# 模拟信号（实际使用时应传入真实数据）
spread_data = {'percentile': 0.85, 'spread': 0.027}
macro_data = {'state': '衰退', 'suggested_position': 0.8}
vol_data = {'vol_zscore': 1.5}

result = timing.evaluate(spread_data, macro_data, vol_data)

print(f'综合评分: {result.composite_score:+.3f}')
print(f'建议仓位: {result.position:.0%}')
print(f'操作: {result.action}')
print(f'解释: {result.explanation}')
print('\n各维度得分:')
for name, score in result.details.items():
    print(f'  {name}: {score:+.3f}')

# ETF配置建议
print('\nETF配置建议:')
allocation = TimingSystem.suggest_etf_allocation(result.position)
for a in allocation:
    if a['权重'] > 0:
        print(f'  {a["ETF"]}({a["代码"]}): {a["权重"]:.0%}')

---
## 总结

本系统从 5 个维度分析红利ETF：

| 模块 | 方法 | 用途 |
|------|------|------|
| 股债利差 | FED Model | 判断安全边际、仓位管理 |
| 宏观状态 | HMM | 识别经济四象限，调整风格暴露 |
| 波动率 | GARCH(1,1) | 捕捉极端事件，恐慌加仓 |
| 风险管理 | VaR/ES/EVT | 量化尾部风险，防止黑天鹅 |
| 综合择时 | 加权集成 | 融合多信号，统一决策 |

### 纪律三原则
1. **信号驱动** - 不凭感觉，严格按系统信号执行
2. **仓位管理** - 永远把风险控制放在第一位
3. **持续迭代** - 用数据验证假设，用回测检验策略